# Gás do Povo — DAG para Avaliação de Impacto (v2)

Este notebook constrói o Directed Acyclic Graph (DAG) metodológico para uma **futura avaliação de impacto** do programa Gás do Povo.

**Correções v2 (após auditoria metodológica):**
- Removido o nó "Dados adicionais necessários": em um DAG formal (Pearl 2009), cada nó representa uma variável aleatória do Processo Gerador de Dados (DGP). Disponibilidade de dados é um requisito operacional do pesquisador, não uma causa. A informação foi movida para a nota ao pé do gráfico.
- Adicionados confundidores latentes não observados (*U*): preço/mercado local de GLP e fricções de acesso (alfabetização digital, distância ao revendedor), que afetam tanto o tratamento quanto o desfecho.
- Corrigida a ambiguidade do rótulo "renda" que aparecia em três nós com papéis causais distintos (pré-tratamento, mediador e desfecho).

In [ ]:
!pip install matplotlib>=3.8.0 numpy>=1.26.0 --quiet

In [ ]:
# Gera o DAG executando o script (fonte unica de verdade — C2)
# O script gerar_dag_gas_do_povo.py contem toda a logica de desenho.
# Importar aqui evita duplicacao de codigo entre notebook e script.

DAG_PATH = "dag_gas_do_povo.png"

In [ ]:
%run gerar_dag_gas_do_povo.py

from IPython.display import Image
Image(DAG_PATH)

## Interpretacao do DAG v3

### Unidade de analise

O DAG representa a logica causal ao nivel da **familia beneficiaria**. Os dados disponiveis nos notebooks 01 e 02 sao agregados no nivel **municipio x mes** — o que implica risco de falacia ecologica: associacoes observadas entre municipios podem nao refletir as relacoes causais individuais. Nesse nivel, X passa a ser composicao municipal (% chefia feminina, % BPC, renda media per capita), nao caracteristica individual.

### Nos e seus papeis causais

| Simbolo | No | Papel | Variaveis representadas |
|---------|-----|-------|------------------------|
| **X** | Caracteristicas pre-tratamento | Confundidor observado | Renda per cap. pre-trat., CadUnico, BPC, chefia feminina, regiao |
| **E** | Elegibilidade formal | Variavel de acesso (intermediaria) | Renda <= 1/2 SM inscrita no CadUnico |
| **T** | Gas do Povo (voucher) | Tratamento | Recebimento e uso efetivo do voucher |
| **Y** | Desfecho social | Resultado causal de interesse | SAN, saude respiratoria, combustivel limpo |
| **M** | Mecanismos (mediadores) | Mediadores pos-tratamento | Acesso a gas, reducao de biomassa, renda domiciliar liberada |
| **U1** | Mercado local de GLP | Confundidor latente | Preco regional, densidade de revendedores credenciados |
| **U2** | Friccoes de acesso | Confundidor latente | Alfabetizacao digital, smartphone, distancia ao revendedor |

> **Convencao:** bordas solidas = variaveis observadas; bordas tracejadas = latentes.  
> **X->M omitida** para clareza visual: pre-tratamento afeta a linha de base do uso de biomassa e acesso a gas, o que importa em analise de mediacao mas nao altera o ajuste para o efeito total.

### Back-door paths de T para Y

| Caminho | Tipo | Bloqueado por |
|---------|------|---------------|
| T <- E <- X -> Y | Back-door (X confundidor) | Ajustar por X (ou E) |
| T <- U1 -> Y | Back-door latente | IV para U1 |
| T <- U2 -> Y | Back-door latente | IV para U2 |

### Conjunto de ajuste — Backdoor Criterion (Pearl 2009)

Ajustar por **Z = {renda per cap. pre-trat., regiao, BPC, chefia feminina}** bloqueia o back-door via X, sob Ignorabilidade Condicional (Rubin 1974). Os back-doors via U1 e U2 permanecem abertos e requerem variaveis instrumentais:

- **IV para U1:** distancia do municipio ao polo de distribuicao de GLP (caracteristica geografica predeterminada) ou cobertura de revendedores credenciados no primeiro mes do programa (pre-programa). **Nao usar preco do petroleo**: viola exclusao pois petroleo -> U1 -> Y.
- **IV para U2:** indice de conectividade municipal pre-programa ou cobertura de Internet movel (exogeno em relacao ao GLP).

### Estrategias de identificacao

| Desenho | Suposicao-chave | Estimando |
|---------|-----------------|-----------|
| **DiD** | Tendencias paralelas; se escalonado: Callaway-Sant'Anna ou Sun-Abraham (nao TWFE) | ATT |
| **PSM** | Ignorabilidade condicional + sobreposicao (variaveis do CadUnico) | ATT |
| **RDD fuzzy** | Continuidade no corte de 1/2 SM + sem manipulacao da renda declarada | LATE |
| **IV** | Relevancia (F > 10) + exclusao + exogeneidade do instrumento | LATE |

> **ITT vs. TOT:** E (elegibilidade) != T (recebimento efetivo). O RDD no corte de renda estima o ITT (efeito de ser elegivel). Para o efeito sobre os que realmente usaram o voucher (TOT/LATE), e necessario o IV com E como instrumento para T.

### Notas adicionais

- **SUTVA/spillovers:** Em escala nacional, a demanda em massa por gas pode elevar precos locais (T -> U1). Isso viola SUTVA e pode atenuar o efeito estimado em analises municipais. Verificar se a variacao de preco e correlacionada com a intensidade do programa.
- **DAG estatico:** O grafo e uma versao colapsada por periodo. Um painel municipio x ano exige indexar no tempo (Tt, Yt, Yt-1) e considerar efeitos defasados.